# Reinforcement Learning - Monte Carlo Methods Assignment

**Course:** AI424 - Reinforcement Learning

---

| Name | ID | Section |
|---|---|---|
| Yussuf Amhed | 20220385 | Section 1 - OOP Implementation |
| Omar Ez-Eldin | 20220228 | Section 2 - Algorithm Details |
| Moaz Gehad | 20220340 | Sections 3 & 4 - Experiments  |
| Abdelrhman Ibrahim | 20220519 | Section 5 - Conceptual Questions |
| Mahmoud Ehab | 20220457 | Section 5 - MountainCar Extension |

---

## Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from collections import defaultdict
import time


---

# Section 1 - OOP Implementation

**Assigned to: Yussuf Ahmed (20220385)**

---

### 1.1 BlackjackEnvironment

Wraps the Gymnasium Blackjack-v1 environment.

In [2]:
class BlackjackEnvironment:
    """
    Wraps the Gymnasium Blackjack-v1 environment.
    """
    def __init__(self):
        self.env = gym.make('Blackjack-v1', sab=True)
        self.action_space = self.env.action_space

    def reset(self):
        state, info = self.env.reset()
        return state

    def step(self, action):
        state, reward, terminated, truncated, info = self.env.step(action)
        done = terminated or truncated
        return state, reward, done, info

    def sample_action(self):
        return self.action_space.sample()

    def close(self):
        self.env.close()

### 1.2 MCPredictionAgent

Implements First-Visit MC, Every-Visit MC, and Incremental MC prediction.

1. Generate episode:
    Generates a full episode until we reach a terminal state

2. evaluate policy
    Every_visit: use all occurences.
    first_vists: use first occurence only.


In [17]:
class MCPredictionAgent:
    """
    Monte Carlo Prediction for a fixed policy.

    Implements First-Visit, Every-Visit, and Incremental variants
    through two constructor flags: mode and use_incremental.
    """
    def __init__(self, env, gamma=1.0 , mode='first_visit', use_incremental=False):
        self.env: BlackjackEnvironment = env
        self.gamma: float = gamma

        self.mode: str = mode  # 'first_visit' or 'every_visit'
        self.use_incremental: bool = use_incremental

        self.value_function = defaultdict(float)
        self.visit_counts = defaultdict(int)
        self.returns_sum = defaultdict(float)

        # Tracking history for Section 3 experiments
        self.history = defaultdict(list)

    def generate_episode(self, policy):
        episode = []
        state = self.env.reset()
        done = False

        while not done:
            action = policy(state)
            next_state, reward, done, info = self.env.step(action)
            episode.append((state, action, reward))
            state = next_state

        return episode


    def evaluate_policy(self, n_episodes, policy, tracked_states=None):

        for _ in range(n_episodes):

            episode = self.generate_episode(policy)
            G = 0
            visited = set()

            # Iterate backwards through the episode to calculate returns
            for t in range(len(episode) - 1, -1, -1):
                state, action, reward = episode[t]
                G = self.gamma * G + reward

                # always add if it is every_vist
                if self.mode == 'every_visit' or (state not in visited):
                    visited.add(state)
                    self.visit_counts[state] += 1

                    if self.use_incremental:
                        # Incremental Update: V(n+1) = V(n) + 1/n(G - V(n))
                        n = self.visit_counts[state]
                        error = G - self.value_function[state]
                        self.value_function[state] += (1.0 / n) * error

                    else:
                        # Standard Average: Sum(Returns) / Count
                        self.returns_sum[state] += G
                        self.value_function[state] = self.returns_sum[state] / self.visit_counts[state]

            # Post-episode history update for EXPERIMENTS (Section 3)
            if tracked_states:
                for s in tracked_states:
                    self.history[s].append(self.value_function[s])

### 1.3 MCControlAgent

On-policy MC Control with epsilon-greedy exploration.

**epsilon_greedy_policy:**  
- Draw a random number. If it's below e -> return a random action (explore).  
- Otherwise -> return the action with the higher Q value (exploit).  
- Tie-breaking: if Q(s,0) == Q(s,1) sample randomly to avoid bias.

**train loop:**  
- Generate episode using e-greedy.  
- For each first-visited (state, action) pair, update Q(s,a).  
- Every track_interval episodes, snapshot the win/loss/draw rates for the learning curve plot.

In [4]:
class MCControlAgent:
    """
    On-Policy Monte Carlo Control Agent (epsilon-greedy).
    """
    def __init__(self, env, gamma=1.0, epsilon=0.1):
        self.env = env
        self.gamma = gamma
        self.epsilon = epsilon

        self.Q = defaultdict(float)
        self.returns_sum = defaultdict(float)
        self.returns_count = defaultdict(int)

        self.win_rates = []
        self.loss_rates = []
        self.draw_rates = []

    def epsilon_greedy_policy(self, state):

        if np.random.random() < self.epsilon:
            return self.env.sample_action()

        q_stick = self.Q[(state, 0)]
        q_hit = self.Q[(state, 1)]

        if q_stick > q_hit:
            return 0
        elif q_hit > q_stick:
            return 1

        return self.env.sample_action()

    def greedy_policy(self, state):
        return 0 if self.Q[(state, 0)] >= self.Q[(state, 1)] else 1

    def generate_episode(self):
        episode = []
        state = self.env.reset()
        done = False

        while not done:
            action = self.epsilon_greedy_policy(state)
            next_state, reward, done, info = self.env.step(action)
            episode.append((state, action, reward))
            state = next_state

        return episode

    def train(self, n_episodes, track_interval=1000):
        wins = losses = draws = 0

        for i in range(1, n_episodes + 1):
            # generate an episode to get the final reward
            episode = self.generate_episode()
            final_reward = episode[-1][2]

            # check the result
            if final_reward > 0: wins += 1
            elif final_reward < 0: losses += 1
            else: draws += 1

            # update the Q(s, a) using first-visit approach.
            G = 0
            visited = set()

            for t in range(len(episode) - 1, -1, -1):
                state, action, reward = episode[t]
                G = self.gamma * G + reward
                sa = (state, action)

                if sa not in visited:
                    visited.add(sa)
                    self.returns_sum[sa] += G
                    self.returns_count[sa] += 1
                    self.Q[sa] = self.returns_sum[sa] / self.returns_count[sa]

            # tracking W/L/D rates every interval for visualization (section 3)
            if i % track_interval == 0:
                total = wins + losses + draws
                self.win_rates.append(wins / total if total > 0 else 0)
                self.loss_rates.append(losses / total if total > 0 else 0)
                self.draw_rates.append(draws / total if total > 0 else 0)
                wins = losses = draws = 0

### 1.4 OffPolicyEvaluator

Ordinary and Weighted Importance Sampling for off-policy evaluation.

**Importance ratio (W)**
- For our deterministic target policy: $\pi$(a|s) = 1 if a = $\pi$(s), else 0.  
- For the uniform behavior policy: b(a|s) = 0.5.

**Handling W == 0**  
- If at any step the target policy wouldn't have taken the action b took, W collapses to 0.  
- Then if W == 0: break since this episode doesn't contribute towards anything.

In [5]:
class OffPolicyEvaluator:
    def __init__(self, env, gamma=1.0, target_policy=None, behavior_policy=None):
        self.env = env
        self.gamma = gamma
        self.target_policy = target_policy
        self.behavior_policy = behavior_policy

        self.V_ordinary  = defaultdict(float)
        self.V_weighted  = defaultdict(float)
        self.C = defaultdict(float)

    def generate_episode(self):
        episode = []
        state = self.env.reset()
        done = False

        while not done:
            action = self.behavior_policy(state)
            next_state, reward, done, info = self.env.step(action)
            episode.append((state, action, reward))
            state = next_state

        return episode

    def compute_importance_ratio(self, episode):
        IR = 1.0

        for state, action, reward in episode:
            pi_prob = 1.0 if self.target_policy(state) == action else 0.0
            b_prob  = 0.5

            if pi_prob == 0.0:
                return 0.0   # early exit

            IR *= pi_prob / b_prob
        return IR




    def ordinary_importance_sampling(self, n_episodes):

        IR_G_sum = defaultdict(float)
        count = defaultdict(int)

        for _ in range(n_episodes):
            episode = self.generate_episode()
            IR = self.compute_importance_ratio(episode)

            G = 0.0
            for t in range(len(episode) - 1, -1, -1):
                state, action, reward = episode[t]
                G = self.gamma * G + reward

                IR_G_sum[state] += IR * G
                count[state] += 1

        # Final estimates
        for s in count:
            self.V_ordinary[s] = IR_G_sum[s] / count[s]

        return self.V_ordinary



    def weighted_importance_sampling(self, n_episodes):

        IR_G_sum = defaultdict(float)
        IR_sum = defaultdict(float)

        for _ in range(n_episodes):
            episode = self.generate_episode()
            IR = self.compute_importance_ratio(episode)

            if IR == 0.0: # zero-weight episode
                continue

            G = 0.0
            for t in range(len(episode) - 1, -1, -1):
                state, action, reward = episode[t]
                G = self.gamma * G + reward

                IR_G_sum[state] += IR * G
                IR_sum[state]   += IR

        for s in IR_sum:
            if IR_sum[s] > 0: # avoiding 0 division
                self.V_weighted[s] = IR_G_sum[s] / IR_sum[s]

        return self.V_weighted



    def incremental_weighted_update(self, n_episodes):

        for _ in range(n_episodes):
            episode = self.generate_episode()
            IR = self.compute_importance_ratio(episode)

            if IR == 0.0:
                continue

            G = 0.0
            for t in range(len(episode) - 1, -1, -1):
                state, action, reward = episode[t]
                G = self.gamma * G + reward

                self.C[state] += IR
                self.V_weighted[state] += (IR / self.C[state]) * (G - self.V_weighted[state])

        return self.V_weighted

### 1.5 MountainCarContinuousAdapter

Discretization wrapper for MountainCarContinuous-v0.

In [6]:
class MountainCarContinuousAdapter:
    def __init__(self, n_pos_bins=20, n_vel_bins=20, n_action_bins=7):
        self.env = gym.make('MountainCarContinuous-v0')
        self.n_pos_bins = n_pos_bins
        self.n_vel_bins = n_vel_bins
        self.n_action_bins = n_action_bins
        self.pos_bins = np.linspace(-1.2, 0.6, n_pos_bins + 1)
        self.vel_bins = np.linspace(-0.07, 0.07, n_vel_bins + 1)
        self.action_bins = np.linspace(-1.0, 1.0, n_action_bins)
        self.n_states = n_pos_bins * n_vel_bins
        self.n_actions = n_action_bins

    def discretize_state(self, s):
        p = np.clip(np.digitize(s[0], self.pos_bins)-1, 0, self.n_pos_bins-1)
        v = np.clip(np.digitize(s[1], self.vel_bins)-1, 0, self.n_vel_bins-1)
        return (int(p), int(v))

    def reset(self):
        s, _ = self.env.reset()
        return self.discretize_state(s)

    def step(self, a):
        ns, r, term, trunc, info = self.env.step(np.array([self.action_bins[a]]))
        return self.discretize_state(ns), r, term or trunc, info

    def sample_action(self):
        return np.random.randint(0, self.n_actions)

    def close(self):
        self.env.close()

### 1.6 Visualiser

All plotting logic centralised here.

In [7]:
class Visualiser:
    @staticmethod
    def plot_value_heatmap(vf, title='Value Function', usable_ace=False):
        player_range = range(12, 22)
        dealer_range = range(1, 11)
        Z = np.zeros((len(player_range), len(dealer_range)))
        for i, p in enumerate(player_range):
            for j, d in enumerate(dealer_range):
                Z[i, j] = vf.get((p, d, usable_ace), 0)
        plt.figure(figsize=(8, 6))
        im = plt.imshow(Z, aspect='auto', origin='lower', cmap='viridis',
                        extent=[0.5, 10.5, 11.5, 21.5])
        ace_str = 'With' if usable_ace else 'Without'
        plt.title(f'{title} ({ace_str} Usable Ace)')
        plt.xlabel('Dealer Showing')
        plt.ylabel('Player Sum')
        plt.colorbar(im)
        plt.show()

    @staticmethod
    def plot_learning_curves(rates_list, labels, title='Learning Curves'):
        plt.figure(figsize=(10, 5))
        for rates, label in zip(rates_list, labels):
            plt.plot(rates, label=label)
        plt.title(title)
        plt.xlabel('Training Interval')
        plt.ylabel('Win Rate')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    @staticmethod
    def plot_is_comparison(V_ord, V_wt, title='IS Comparison'):
        states = [(20, 10, False), (13, 2, False), (18, 9, True)]
        labels = [str(s) for s in states]
        o = [V_ord.get(s, 0) for s in states]
        w = [V_wt.get(s, 0) for s in states]
        x = np.arange(len(labels))
        width = 0.35
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(x - width/2, o, width, label='Ordinary IS', color='coral')
        ax.bar(x + width/2, w, width, label='Weighted IS', color='teal')
        ax.set_xlabel('State')
        ax.set_ylabel('Estimated Value')
        ax.set_title(title)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=45)
        ax.legend()
        plt.tight_layout()
        plt.show()

#### Section 1 - Verification

In [10]:
env = BlackjackEnvironment()
def simple_policy(state):
    return 0 if state[0] >= 20 else 1
def target_policy(state):
    return 0 if state[0] >= 20 else 1
def behavior_policy(state):
    return np.random.randint(0, 2)

mc_pred = MCPredictionAgent(env, gamma=1.0)
mc_ctrl = MCControlAgent(env, gamma=1.0, epsilon=0.1)
off_eval = OffPolicyEvaluator(env, gamma=1.0, target_policy=target_policy, behavior_policy=behavior_policy)
vis = Visualiser()
print('All classes instantiated successfully.')

All classes instantiated successfully.


---

# Section 2 - Algorithm Details

**Assigned to: Omar Ez-Eldin (20220228)**

---

### 2.1 First-Visit MC Prediction

In [18]:
tracked = [(20, 10, False), (13, 2, False), (18, 9, True)]

env = BlackjackEnvironment()

mc_fv = MCPredictionAgent(
    env,
    gamma=1.0,
    mode='first_visit',
    use_incremental=False
)

start = time.time()
mc_fv.evaluate_policy(10000, policy=simple_policy, tracked_states=tracked)

print(f'First-Visit MC (10k eps): {time.time()-start:.4f}s')
print(f'States evaluated: {len(mc_fv.value_function)}')

for s in tracked:
    val = mc_fv.value_function.get(s, 0)
    print(f'  V{s} = {val:.4f}')

print(f'History points for {tracked[0]}: {len(mc_fv.history[tracked[0]])}')

First-Visit MC (10k eps): 1.2286s
States evaluated: 280
  V(20, 10, False) = 0.4380
  V(13, 2, False) = -0.5765
  V(18, 9, True) = -0.4211
History points for (20, 10, False): 10000


### 2.2 Every-Visit MC Prediction

In [20]:
env2 = BlackjackEnvironment()

mc_ev = MCPredictionAgent(
    env2,
    gamma=1.0,
    mode='every_visit',
    use_incremental=False
)

# Run Evaluation
start = time.time()
mc_ev.evaluate_policy(10000, policy=simple_policy)

# Results
print(f'Every-Visit MC (10k eps): {time.time()-start:.4f}s')
for s in [(20,10,False), (13,2,False), (18,9,True)]:
    val = mc_ev.value_function.get(s, 0)
    print(f'  V{s} = {val:.4f}')

Every-Visit MC (10k eps): 1.9988s
  V(20, 10, False) = 0.4481
  V(13, 2, False) = -0.5667
  V(18, 9, True) = -0.4286


## 2.3 Incremental first-visit and every-vsist

In [22]:

print("Incremental First-Visit MC")
env_inc_fv = BlackjackEnvironment()
mc_inc_fv = MCPredictionAgent(
    env_inc_fv,
    mode='first_visit',
    use_incremental=True
)

start_time = time.time()
mc_inc_fv.evaluate_policy(10000, policy=simple_policy)
duration = time.time() - start_time

print(f'Incremental First-Visit MC (10k eps): {duration:.4f}s')
for s in [(20, 10, False), (13, 2, False), (18, 9, True)]:
    val = mc_inc_fv.value_function.get(s, 0)
    print(f'  V{s} = {val:.4f}')

Incremental First-Visit MC
Incremental First-Visit MC (10k eps): 1.2395s
  V(20, 10, False) = 0.4932
  V(13, 2, False) = -0.5865
  V(18, 9, True) = -0.4000


In [23]:
print("Testing Incremental Every-Visit MC")
env_inc_ev = BlackjackEnvironment()
mc_inc_ev = MCPredictionAgent(
    env_inc_ev,
    mode='every_visit',
    use_incremental=True
)

start_time = time.time()
mc_inc_ev.evaluate_policy(10000, policy=simple_policy)
duration = time.time() - start_time

print(f'Incremental Every-Visit MC (10k eps): {duration:.4f}s')
for s in [(20, 10, False), (13, 2, False), (18, 9, True)]:
    val = mc_inc_ev.value_function.get(s, 0)
    print(f'  V{s} = {val:.4f}')

Testing Incremental Every-Visit MC
Incremental Every-Visit MC (10k eps): 1.2678s
  V(20, 10, False) = 0.3926
  V(13, 2, False) = -0.6771
  V(18, 9, True) = -0.6000


### 2.4 On-Policy MC Control

In [24]:
env3 = BlackjackEnvironment()
mc_control = MCControlAgent(env3, gamma=1.0, epsilon=0.1)
start = time.time()
mc_control.train(100000, track_interval=10000)
print(f'MC Control (100k eps, eps=0.1): {time.time()-start:.4f}s')
print(f'Q-values: {len(mc_control.Q)}')
print(f'Win rates: {[f"{r:.3f}" for r in mc_control.win_rates]}')

MC Control (100k eps, eps=0.1): 14.1616s
Q-values: 560
Win rates: ['0.391', '0.400', '0.409', '0.406', '0.417', '0.412', '0.414', '0.415', '0.417', '0.421']


### 2.5 Off-Policy MC Evaluation

In [ ]:

env_off = BlackjackEnvironment()
off_pol = OffPolicyEvaluator(
    env_off,
    gamma=1.0,
    target_policy=target_policy,    # stick on 20+
    behavior_policy=behavior_policy # uniform random p = 0.5
)

tracked = [(20, 10, False), (13, 2, False), (18, 9, True)]


print(f"{'-'*10}Running Ordinary IS{'-'*10}")
start = time.time()
off_pol.ordinary_importance_sampling(n_episodes=50_000)
print(f"Done: {time.time()-start:.2f}s\n")

off_pol.V_weighted = defaultdict(float)
off_pol.C = defaultdict(float)

print(f"{'-'*10}Running Weighted IS{'-'*10}")
start = time.time()
off_pol.weighted_importance_sampling(n_episodes=50_000)
print(f"Done: {time.time()-start:.2f}s\n")


off_pol_inc = OffPolicyEvaluator(
    env_off,
    gamma=1.0,
    target_policy=target_policy,
    behavior_policy=behavior_policy
)

print(f"{'-'*10}Running Incremental Weighted IS{'-'*10}")
start = time.time()
off_pol_inc.incremental_weighted_update(n_episodes=50_000)
print(f"Done: {time.time()-start:.2f}s\n")

# ------------------- results -----------------
print(f"{'State':<30} {'Ordinary IS':>15}  {'Weighted IS':>15}  {'Incremental IS':>15}")

for s in tracked:
    o = off_pol.V_ordinary.get(s, 0)
    w = off_pol.V_weighted.get(s, 0)
    inc = off_pol_inc.V_weighted.get(s, 0)
    print(f"{str(s):<30} {o:>15.4f}  {w:>15.4f}  {inc:>15.4f}")



print("\nDifference between Weighted and Incremental Weighted IS:")
for s in tracked:
    w = off_pol.V_weighted.get(s, 0)
    inc = off_pol_inc.V_weighted.get(s, 0)
    print(f"{str(s)}: {abs(w - inc):.6f}  (should be near 0)")

----------Running Ordinary IS----------
Done: 7.10s

----------Running Weighted IS----------
Done: 6.02s

----------Running Incremental Weighted IS----------


---

# Section 3 - Experiments

**Assigned to: Moaz Gehad (20220340)**

---

### 3.1 Prediction Experiment

### 3.2 On-Policy Control Experiment

### 3.3 Off-Policy Evaluation Experiment

---

# Section 4 - Conceptual Questions

**Assigned to: Abdelrhman Ibrahim (20220519)**

---

### Q1: Why does MC require episode termination?

TO DO

### Q2: First-Visit vs Every-Visit MC
TO DO

### Q3: Ordinary vs Weighted Importance Sampling

TO DO

### Q4: DP vs MC

TO DO

### Q5: MC for non-episodic tasks

TO DO

### Q6: Why epsilon-greedy matters

TO DO

### Q7: Effect of discount factor

TO DO

### Q8: MC in high-dimensional spaces

TO DO

### Q9: Model-based vs Model-free trade-offs

TO DO

### Q10: MC convergence guarantees

TO DO

---

# Section 5 - MountainCar Extension

**Assigned to: Mahmoud Ehab (20220457)**

---

### 5.1 Environment Exploration

### 5.2 MC Control on MountainCar

### 5.3 Analysis



## Conclusion

This assignment demonstrated Monte Carlo methods for model-free RL:

- **Sections 1-2**: Implemented and validated First-Visit MC, Every-Visit MC, Incremental MC, On-Policy Control, and Off-Policy Evaluation on Blackjack
- **Section 3**: More episodes improve estimates; epsilon controls exploration-exploitation; Weighted IS is more stable
- **Section 4**: Answered conceptual questions on MC theory
- **Section 5**: MC struggles with continuous environments due to sparse rewards and discretization errors

**Key Lesson**: MC methods excel for episodic tasks with manageable state spaces. Continuous control requires more sophisticated techniques.